## Notebook conventions

**Edit this file in place — don't save a new copy** (no `_V1`/`_V2`/dated/`-Copy`/`DEBUG-` variants). Commit changes via a branch + PR; git history is the version record, not the filename. Full conventions and `nbstripout` setup: see the repo [README](../../README.md) → "Notebook conventions."

# Video Processing Notebook for MegaDetector

This notebook extracts frames from video files and processes them through MegaDetector for wildlife detection.

**System Requirements:**
- Windows OS
- Python 3.8+
- MegaDetector installed
- OpenCV

In [ ]:
import os
import json
import shutil
import subprocess
import time
from pathlib import Path
import cv2
import tkinter as tk
from tkinter import filedialog as fd

# MegaDetector imports (from the pip-installed package)
from megadetector.utils import path_utils
from megadetector.detection import video_utils
from megadetector.utils.ct_utils import write_json
from megadetector.visualization import visualization_utils as vis_utils
from megadetector.detection.video_utils import frame_results_to_video_results, FrameToVideoOptions

# Create a hidden root window for file dialogs
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

print("All imports successful.")


## Configuration

Set up paths and detection parameters.

In [ ]:
# Model configuration
#
# Pointing directly at a shared, machine-wide model file instead of the "MDV5A"
# shorthand. The shorthand triggers an auto-download to each Windows profile's
# per-user %TEMP% cache the first time that profile runs the notebook -- this is
# what caused a CERTIFICATE_VERIFY_FAILED error under Shawn's account (the model
# was never cached there). Pointing to an explicit path skips the per-user cache
# and the download entirely, for every user, every time.
# See Tarazed-Session-Log.docx for the full diagnostic trail.
MODEL_PATH = r"C:\ProgramData\megadetector_models\md_v5a.0.1.pt"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Shared model file not found at {MODEL_PATH}. "
        "This notebook expects the model to already be staged at this shared "
        "location -- see Tarazed-Session-Log.docx for setup steps."
    )

MODEL_NAME = MODEL_PATH  # kept as MODEL_NAME so the rest of the notebook is unchanged

# Detection parameters
ANIMAL_CATEGORY_ID = 1
CONF_THRESH = 0.2  # Confidence threshold (0.0 to 1.0)

# Verify MegaDetector is importable from the active environment
import megadetector  # noqa: E402
print(f"MegaDetector found: {megadetector.__file__}")
print(f"Model: {MODEL_NAME}")
print(f"Confidence threshold: {CONF_THRESH}")

## Select Directories

Choose your input folder (containing videos) and output folder (where results will be saved).

In [ ]:
# Select input folder containing video files
input_folder = fd.askdirectory(title="Select folder containing video files")
if not input_folder:
    raise ValueError("No input folder selected")

DATA_DIR = Path(input_folder)
print("Input folder:", DATA_DIR)

In [ ]:
# Select output folder for all results
output_folder = fd.askdirectory(title="Select folder for output (detections, videos, etc.)")
if not output_folder:
    raise ValueError("No output folder selected")

OUTPUT_DIR = Path(output_folder)
OUTPUT_DIR.mkdir(exist_ok=True)
print("Output folder:", OUTPUT_DIR)

## Helper Functions

In [ ]:
def find_all_videos(root):
    """Find all video files recursively."""
    exts = (".mp4", ".mov", ".avi", ".MP4", ".MOV", ".AVI")
    videos = []
    for r, _, files in os.walk(root):
        for f in files:
            if f.endswith(exts):
                videos.append(Path(r) / f)
    return sorted(videos)

all_videos = find_all_videos(DATA_DIR)
print(f"Found {len(all_videos)} videos")

# Show first 20 videos
for v in all_videos[:20]:
    print(f"  {v.name}")
    
if len(all_videos) > 20:
    print(f"  ... and {len(all_videos) - 20} more")

In [ ]:
def resolve_frame_path(frames_dir, file_field):
    """Resolve the path to a frame file from MegaDetector JSON."""
    p = Path(file_field)
    if p.is_absolute() or p.exists():
        return p
    return frames_dir / p.name

## Process Videos

This cell processes each video individually:
1. Extract frames
2. Run MegaDetector
3. Draw bounding boxes on frames with detections
4. Create output video (only for videos WITH animal detections)
5. Clean up temporary frames

**Note:** You can test with a few videos first by uncommenting the test line in the loop.

In [ ]:
import sys
print(sys.executable)

In [ ]:
# helper: check if video has any animal detections (any frame, whole video)
def video_has_any_animal(images, conf_thresh=0.0):
    for img in images:
        for det in img.get("detections", []):
            if int(det["category"]) == ANIMAL_CATEGORY_ID and det["conf"] >= conf_thresh:
                return True
    return False

# helper: check if a single frame has an animal detection (used to filter frames when restitching video)
def frame_has_animal(img, conf_thresh):
    for det in img.get("detections", []):
        if int(det["category"]) == ANIMAL_CATEGORY_ID and det["conf"] >= conf_thresh:
            return True
    return False

batch_start = time.perf_counter()

videos_processed = 0
videos_with_animals = 0
videos_skipped = 0
videos_with_animals_list = []
videos_skipped_list = []

for video_path in all_videos:
#for video_path in all_videos[:3]:  # TESTING: Uncomment to process only first 3 videos

    print("\n" + "="*50)
    print("Processing:", video_path.name)
    print("="*50)

    # ----------------------------------------
    # Per-video output paths
    # ----------------------------------------
    video_stem = video_path.stem
    video_out_dir = OUTPUT_DIR / video_stem

    frames_dir = video_out_dir / "frames"
    json_path = video_out_dir / "detections.json"
    out_video = video_out_dir / f"{video_stem}_detected.mp4"

    video_out_dir.mkdir(parents=True, exist_ok=True)

    print("Frames:", frames_dir)
    print("JSON:", json_path)
    print("Output:", out_video)

    # ----------------------------------------
    # Clean frames dir
    # ----------------------------------------
    if frames_dir.exists():
        shutil.rmtree(frames_dir)
    frames_dir.mkdir()

    # ----------------------------------------
    # Extract frames
    # ----------------------------------------
    print("\n[1/4] Extracting frames...")
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        cv2.imwrite(
            str(frames_dir / f"frame_{frame_idx:05d}.jpg"),
            frame
        )
        frame_idx += 1

    cap.release()
    print(f"  Extracted {frame_idx} frames @ {fps:.2f} FPS")

    if frame_idx == 0:
        print("  No frames extracted, skipping video")
        continue

    # ----------------------------------------
    # Run MegaDetector (pip-installed package, invoked as a module)
    # ----------------------------------------
    print("\n[2/4] Running MegaDetector...")
    cmd = [
        "python",
        "-m", "megadetector.detection.run_detector_batch",
        MODEL_NAME,
        str(frames_dir),
        str(json_path),
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("  MegaDetector failed:")
        print(result.stderr[-2000:])  # last chunk, tracebacks are usually at the end
        shutil.rmtree(frames_dir)
        videos_skipped += 1
        videos_processed += 1
        videos_skipped_list.append(video_path.name)
        continue
    print("  Detection complete")

    # ----------------------------------------
    # Load detections
    # ----------------------------------------
    with open(json_path) as f:
        md = json.load(f)

    images = md.get("images", [])
    print(f"  Loaded {len(images)} detection results")

    if len(images) != frame_idx:
        print(f"  Frame/JSON mismatch ({frame_idx} frames vs {len(images)} results), skipping")
        shutil.rmtree(frames_dir)
        continue

    # --------------------------------------------------
    # NEW: Skip videos with NO animal detections
    # --------------------------------------------------
    print("\n[3/4] Checking for animal detections...")
    if not video_has_any_animal(images, CONF_THRESH):
        print(f"  No animal detections (conf >= {CONF_THRESH}) -- skipping output video")
        shutil.rmtree(frames_dir)
        videos_skipped += 1
        videos_processed += 1
        videos_skipped_list.append(video_path.name)
        continue

    print("  Animals detected.")

    # ----------------------------------------
    # Initialize output video
    # ----------------------------------------
    print("\n[4/4] Creating output video (animal frames only, with bounding boxes)...")
    first_frame_path = resolve_frame_path(frames_dir, images[0]["file"])
    first_frame = cv2.imread(str(first_frame_path))

    if first_frame is None:
        print("  Cannot read first frame, skipping")
        continue

    h, w, _ = first_frame.shape

    out = cv2.VideoWriter(
        str(out_video),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h),
    )

    # --------------------------------------------------
    # NEW: Skip frames with NO animal detections
    # (only frames that pass this check get written below --
    # this is what makes the output video an animal-only
    # highlight reel instead of the full original video)
    # --------------------------------------------------
    written = 0
    frames_skipped_no_animal = 0
    detections_drawn = 0

    for img in images:
        if not frame_has_animal(img, CONF_THRESH):
            frames_skipped_no_animal += 1
            continue

        frame_path = resolve_frame_path(frames_dir, img["file"])
        frame = cv2.imread(str(frame_path))
        if frame is None:
            frames_skipped_no_animal += 1
            continue

        # draw bounding boxes (animal only)
        for det in img.get("detections", []):
            if int(det["category"]) != ANIMAL_CATEGORY_ID:
                continue
            if det["conf"] < CONF_THRESH:
                continue

            x, y, bw, bh = det["bbox"]
            x1 = int(x * w)
            y1 = int(y * h)
            x2 = int((x + bw) * w)
            y2 = int((y + bh) * h)

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                frame,
                f"{det['conf']:.2f}",
                (x1, max(y1 - 5, 10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 0),
                1,
            )
            detections_drawn += 1

        out.write(frame)
        written += 1

    out.release()
    print(f"  Wrote {written} animal frames with {detections_drawn} bounding boxes")
    print(f"  Skipped {frames_skipped_no_animal} frames with no animal detection")
    print(f"  Output video: {out_video}")

    videos_with_animals += 1
    videos_processed += 1
    videos_with_animals_list.append(video_path.name)

    # ----------------------------------------
    # Cleanup frames
    # ----------------------------------------
    shutil.rmtree(frames_dir)
    print("  Cleaned up temporary frames")

print("\n" + "="*50)
print("PROCESSING COMPLETE")
print("="*50)


## Processing Summary

In [ ]:
batch_end = time.perf_counter()
elapsed = batch_end - batch_start

print("\n" + "="*50)
print("BATCH PROCESSING SUMMARY")
print("="*50)
print(f"Total videos found     : {len(all_videos)}")
print(f"Videos processed       : {videos_processed}")
print(f"Videos with animals    : {videos_with_animals}")
print(f"Videos skipped (empty) : {videos_skipped}")
print(f"Total elapsed time     : {elapsed:.2f} seconds ({elapsed/60:.1f} minutes)")
print(f"Average per video      : {elapsed / max(videos_processed, 1):.2f} seconds")

print("\n" + "-"*50)
print("Videos WITH animal detections:")
print("-"*50)
if videos_with_animals_list:
    for v in videos_with_animals_list:
        print(f"  - {v}")
else:
    print("  (none)")

print("\n" + "-"*50)
print("Videos with NO animal detections:")
print("-"*50)
if videos_skipped_list:
    for v in videos_skipped_list:
        print(f"  - {v}")
else:
    print("  (none)")

print("\n" + "="*50)
print(f"Output directory: {OUTPUT_DIR}")
print("="*50)
